En este proyecto vamos a trabajar con un caso ficticio donde tenemos una base de datos de unos 500 registros financieros

In [20]:
import numpy as np
import pandas as pd

# Fijamos semilla para consistencia de datos
np.random.seed(101)

# 1. Generación masiva de datos (500 transacciones de alta frecuencia)
num_registros = 500
activos = np.random.choice(['QUANT-X', 'NEXUS-Tech', 'AERO-V', 'CYBER-9'], size=num_registros)
volumenes = np.random.randint(1000, 50000, size=num_registros)
precios_raw = np.random.uniform(15.0, 850.0, size=num_registros).astype(np.float32)

# 2. Inyección de Anomalías (25 cortes de energía aleatorios en la API del Nasdaq)
indices_corruptos = np.random.choice(num_registros, size=25, replace=False)
precios_raw[indices_corruptos] = np.nan

# 3. Construcción del DataFrame del Mercado
df_mercado = pd.DataFrame({
    'Activo_Financiero': activos,
    'Volumen_Transado': volumenes,
    'Precio_Cierre': precios_raw
})

print("¡Feed de mercado conectado! Filas totales para procesar:", len(df_mercado))

¡Feed de mercado conectado! Filas totales para procesar: 500


Veamos cuantas filas contienen datos fallidos/erroneos

In [21]:
filas_erroneas = df_mercado[df_mercado['Precio_Cierre'].isna()] # con este trozo de codigo encontramos cuantas filas erroneas hay
print("Número de registros con precios corruptos (NaN):", len(filas_erroneas))

mediana_precio = df_mercado['Precio_Cierre'].median() # calculamos la mediana de los precios para reemplazar los valores corruptos
#creamos una copia del dataframe original para no modificarlo directamente
df_mercado_limpio = df_mercado.copy() # creamos un nuevo dataframe solo con los registros válidos para calcular la mediana
df_mercado_limpio['Precio_Cierre'] = df_mercado_limpio['Precio_Cierre'].fillna(mediana_precio) # reemplazamos los valores corruptos con la mediana

#Ahora volvemos a contar cuantas filas erroneas hay para verificar que se han corregido
filas_erroneas_post_correccion = df_mercado_limpio[df_mercado_limpio['Precio_Cierre'].isna()]
print("Número de registros con precios corruptos después de la corrección:", len(filas_erroneas_post_correccion))

Número de registros con precios corruptos (NaN): 25
Número de registros con precios corruptos después de la corrección: 0


Una vez corregido el problema de los datos corruptos, ahora vamos a calcular el precio de las operaciones

In [22]:
df_mercado_limpio['Valor_Operacion_USD'] = df_mercado_limpio['Volumen_Transado'] * df_mercado_limpio['Precio_Cierre']
print(df_mercado_limpio)

    Activo_Financiero  Volumen_Transado  Precio_Cierre  Valor_Operacion_USD
0             CYBER-9             25262     282.988586         7.148858e+06
1             CYBER-9              8439     630.073181         5.317188e+06
2          NEXUS-Tech             10436      47.771584         4.985442e+05
3              AERO-V              9594     330.544586         3.171245e+06
4             CYBER-9             26793     811.254639         2.173595e+07
..                ...               ...            ...                  ...
495            AERO-V             35859     523.777954         1.878215e+07
496           QUANT-X             29974     345.266602         1.034902e+07
497           CYBER-9             38086     212.632278         8.098313e+06
498           CYBER-9             46380     831.276489         3.855460e+07
499        NEXUS-Tech             48031      77.488670         3.721858e+06

[500 rows x 4 columns]


Ahora vamos a hacer codigo para encontrar 'Ballenas' y dsitinguirlas de operaciones normales

In [23]:
df_mercado_limpio['Alerta_Ballena'] = np.where((df_mercado_limpio['Precio_Cierre'] > 600)&(df_mercado_limpio['Volumen_Transado']>40000), 'BALLENA DETECTADA', 'OPERACION NORMAL')
print(df_mercado_limpio)

    Activo_Financiero  Volumen_Transado  Precio_Cierre  Valor_Operacion_USD  \
0             CYBER-9             25262     282.988586         7.148858e+06   
1             CYBER-9              8439     630.073181         5.317188e+06   
2          NEXUS-Tech             10436      47.771584         4.985442e+05   
3              AERO-V              9594     330.544586         3.171245e+06   
4             CYBER-9             26793     811.254639         2.173595e+07   
..                ...               ...            ...                  ...   
495            AERO-V             35859     523.777954         1.878215e+07   
496           QUANT-X             29974     345.266602         1.034902e+07   
497           CYBER-9             38086     212.632278         8.098313e+06   
498           CYBER-9             46380     831.276489         3.855460e+07   
499        NEXUS-Tech             48031      77.488670         3.721858e+06   

        Alerta_Ballena  
0     OPERACION NORMAL  
1

Ahora vamos a agrupar los datos para tenerlos mas ordenados y que sea mas facil de verlos y analizarlos

In [25]:
df_mercado_limpio.groupby('Activo_Financiero').agg({
    'Valor_Operacion_USD': 'sum',
    'Precio_Cierre': 'mean',
    'Volumen_Transado': 'max'
})

,Valor_Operacion_USD,Precio_Cierre,Volumen_Transado
Activo_Financiero,,,
AERO-V,1.309632e+09,396.741638,49561
CYBER-9,1.753503e+09,461.310760,49618
NEXUS-Tech,1.226652e+09,416.017059,49394
QUANT-X,1.321868e+09,432.454987,49430
